In [1]:
!pip install scikit-learn


In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load recipe data from Excel
file_path = "groundfinal.xlsx"  # Upwhdate the file name if needed
recipe_df = pd.read_excel(file_path)

# Display the first few rows to confirm
print(recipe_df.head())

# Combine columns into a single string for each recipe
recipe_df['combined_text'] = recipe_df['Ingredients'] + " | " + recipe_df['Cuisine Region'] + " | " + recipe_df['Cooking Method'] + " | " + recipe_df['Recipe']

# Convert the combined text to a list
recipes = recipe_df['combined_text'].tolist()

# Encode all recipes into embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')  # Use the SentenceTransformer model of your choice
recipe_embeddings = model.encode(recipes)

C:\Users\user\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



           Recipe Name                                        Ingredients  \
0                Pulao  Rice, Mutton/Beef, Whole Spices, Onions, Ginge...   
1               Tahari  Rice, Potatoes, Onions, Tomatoes, Green Chilie...   
2          Saag Chawal  Rice, Spinach, Mustard Greens, Green Chilies, ...   
3         Machli Pulao  Rice, Fish, Yogurt, Tomatoes, Onions, Green Ch...   
4  Chicken Qorma Pulao  Rice, Chicken, Yogurt, Qorma Masala, Onions, G...   

  Cuisine Region      Cooking Method  \
0         Punjab  Boiling, Simmering   
1         Punjab     One-Pot Cooking   
2         Punjab   Boiling, Sautéing   
3          Sindh  Boiling, Simmering   
4         Punjab  Boiling, Simmering   

                                              Recipe Category  
0  Sauté onions and whole spices in ghee. Add mea...     Rice  
1  Sauté onions and spices in ghee. Add potatoes ...     Rice  
2  Boil rice separately. Sauté spinach and mustar...     Rice  
3  Cook fish with spices and tomatoes. 

In [3]:
print(recipe_embeddings.shape)  # Should be (number_of_recipes, embedding_dimension)
print(recipe_embeddings[0])     # View the embedding for the first recipe


(3305, 384)
[-6.71383664e-02 -1.07581895e-02 -2.61787027e-02  7.57943913e-02
 -1.14311211e-01  5.36425337e-02 -3.14057246e-02 -8.69567096e-02
  5.91023732e-03 -6.39985576e-02  9.90367457e-02 -7.82408565e-02
 -1.35887237e-02 -5.35124950e-02  4.02864031e-02 -5.13305925e-02
  1.15933239e-01  4.76292893e-03 -2.12602429e-02 -1.56576574e-01
 -4.15239558e-02 -4.08815360e-03  2.33754013e-02 -3.56671736e-02
 -1.29412897e-02 -3.00891027e-02  4.38455716e-02  2.06905305e-02
 -1.78965572e-02 -4.27860469e-02  4.23068665e-02 -2.78890859e-02
  3.29848044e-02  4.96374555e-02 -2.88819745e-02  1.19067812e-02
 -2.01742891e-02 -4.15171906e-02  7.75361061e-02  7.01771230e-02
  8.86630490e-02  5.98718897e-02  5.29693589e-02  7.68508250e-03
  5.86240441e-02  1.86700989e-02 -3.84730957e-02 -8.74107424e-03
  6.85224123e-03 -5.89784794e-02 -3.49703282e-02 -2.61038151e-02
 -4.33076210e-02 -2.51116673e-03  6.82718083e-02 -8.66372064e-02
 -5.89726865e-02 -3.78785692e-02  3.01038921e-02  5.36629930e-02
 -1.84411202e

In [4]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Example function to handle multi-label matching in categories
def matches_category(selected_category, category_field):
    # Check if the selected category is part of the category field (multi-label)
    return any(selected_category in cat for cat in category_field.split('/'))

# Get ingredients from the user
user_input = input("Enter the ingredients you have (comma-separated): ").strip()

# Generate embedding for user input (using your model, assumed to be pre-loaded)
user_embedding = model.encode([user_input])

# Compute similarity scores (this assumes you have pre-computed recipe embeddings)
similarities = cosine_similarity(user_embedding, recipe_embeddings)

# Get available categories and cuisine regions
available_categories = recipe_df['Category'].unique()
available_cuisines = recipe_df['Cuisine Region'].unique()

# Display available categories and cuisines
print("\nAvailable Categories:")
for category in available_categories:
    print(f"- {category}")

print("\nAvailable Cuisine Regions:")
for cuisine in available_cuisines:
    print(f"- {cuisine}")

# Ask user if they want to narrow down the search by category or cuisine region
category_choice = input("\nWould you like to narrow the search by category? (yes/no): ").strip().lower()
cuisine_choice = input("Would you like to narrow the search by cuisine region? (yes/no): ").strip().lower()

# Filter based on category if user chooses to
if category_choice == 'yes':
    selected_category = input(f"Please choose a category from the above options: ").strip()
    recipe_df = recipe_df[recipe_df['Category'].apply(lambda x: matches_category(selected_category, x))]

# Filter based on cuisine region if user chooses to
if cuisine_choice == 'yes':
    selected_cuisine = input(f"Please choose a cuisine region from the above options: ").strip()
    recipe_df = recipe_df[recipe_df['Cuisine Region'] == selected_cuisine]

# Combine columns again for the filtered recipes
filtered_recipes = recipe_df['Ingredients'] + " | " + recipe_df['Cuisine Region'] + " | " + recipe_df['Cooking Method'] + " | " + recipe_df['Recipe']
filtered_recipe_embeddings = model.encode(filtered_recipes.tolist())

# Compute similarity scores with the filtered recipes
similarities = cosine_similarity(user_embedding, filtered_recipe_embeddings)

# Find the most similar recipe
most_similar_index = similarities[0].argmax()
print("\nBest Match Recipe Name:", recipe_df.iloc[most_similar_index]['Recipe Name'])

# Display top-N matches
top_n = 3
top_indices = similarities[0].argsort()[-top_n:][::-1]  # Sort to get top N indices in descending order

print("\nTop Matches:")
for idx in top_indices:
    recipe_info = recipe_df.iloc[idx]
    print(f"Recipe Name: {recipe_info['Recipe Name']} (Similarity: {similarities[0][idx]:.2f})")
    print(f"  Ingredients: {recipe_info['Ingredients']}")
    print(f"  Cuisine Region: {recipe_info['Cuisine Region']}")
    print(f"  Cooking Method: {recipe_info['Cooking Method']}")
    print(f"  Recipe: {recipe_info['Recipe']}")
    print(f"  Category: {recipe_info['Category']}\n")


Enter the ingredients you have (comma-separated):  Beef,Black Pepper,Garlic,Ghee



Available Categories:
- Rice
- Meat
- Vegetarian
- Street Food
- Dessert
- BBQ-GRILL
- Seafood
- Breakfast
- Drinks
- Rice/Vegetarian
- Breakfast/Street Food
- Meat/Rice
- Meat/Street Food
- Breakfast/Rice
- Rice/Street Food
- Breakfast/Meat/Rice
- Breakfast/Drinks
- Rice/Street Food/Vegetarian
- BBQ-GRILL/Street Food
- Dessert/Drinks
- Breakfast/Rice/Vegetarian
- Dessert/Drinks/Street Food
- Seafood/Street Food
- Rice/Seafood/Street Food
- BBQ-GRILL/Seafood
- Dessert/Street Food
- Breakfast/Dessert/Street Food
- Breakfast/Dessert
- Breakfast/Drinks/Street Food
- Breakfast/Vegetarian
- Rice/Seafood
- Breakfast/Seafood
- Dessert/Rice

Available Cuisine Regions:
- Punjab
- Sindh
- Kashmir
- Khyber Pakhtunkhwa
- Coastal Region
- Hyderabad
- Fusion
- Balochistan



Would you like to narrow the search by category? (yes/no):  no
Would you like to narrow the search by cuisine region? (yes/no):  no



Best Match Recipe Name: Beef Namkeen Gosht

Top Matches:
Recipe Name: Beef Namkeen Gosht (Similarity: 0.73)
  Ingredients: Beef, black pepper, garlic, ginger, salt, ghee
  Cuisine Region: Balochistan
  Cooking Method: Grilling
  Recipe: Beef is marinated with black pepper and salt, grilled over a flame, and served with naan or roti.
  Category: Meat

Recipe Name: Dum Pukht (Similarity: 0.71)
  Ingredients: Beef, yogurt, garlic, garam masala, cinnamon, cardamom, ghee
  Cuisine Region: Balochistan
  Cooking Method: Slow cooking
  Recipe: Cook beef with spices on low heat until tender.
  Category: BBQ-GRILL

Recipe Name: BBQ Beef Burgers (Similarity: 0.68)
  Ingredients: Ground beef, garlic, ginger, cumin, coriander, black pepper
  Cuisine Region: Sindh
  Cooking Method: BBQ
  Recipe: Shape beef patties, BBQ until done, serve in buns.
  Category: BBQ-GRILL



In [50]:
test_queries = [
    {
        "user_input": "Beef, Black Pepper, Garlic, Ghee",
        "relevant_recipes": [
            "Beef Namkeen Gosht",
            "Dum Pukht",
            "Beef Pasanda Biryani",
            "Black Pepper Pulao",
            "Spicy Beef Biryani",
            "Beef Haleem Rice",
            "Beef and Lentil Rice",
            "Beef and Potato Pulao"
        ]
    },
    {
        "user_input": "Chicken, Yogurt, Spices, Garlic",
        "relevant_recipes": [
            "Chicken Qorma Pulao",
            "Achari Chicken Biryani",
            "Karachi Chicken Biryani",
            "Chicken Makhani Biryani",
            "Methi Chicken Biryani",
            "Chicken Biryani with Mint",
            "Chicken Keema Pulao",
            "Tikka Biryani",
            "Chicken Nihari Rice"
        ]
    },
    {
        "user_input": "Mutton, Cumin, Garlic, Onion",
        "relevant_recipes": [
            "Mutton Karahi Pulao",
            "Peshawari Mutton Pulao",
            "Kachchi Gosht Biryani",
            "Mutton Korma Biryani",
            "Bhuna Gosht Pulao",
            "Mutton Pulao with Yogurt",
            "Mutton Yakhni Pulao",
            "Mutton Shank Pulao",
            "Mutton and Vegetable Pulao"
        ]
    },
    {
        "user_input": "Fish, Coconut, Lemon, Spices",
        "relevant_recipes": [
            "Coconut Fish Pulao",
            "Tamarind Fish Biryani",
            "Fish Zafrani",
            "Lemon Shrimp Pulao",
            "Coconut Prawn Biryani",
            "Coconut Chicken Pulao",
            "Shrimp and Tomato Rice"
        ]
    },
    {
        "user_input": "Vegetables, Cumin, Ghee, Garlic",
        "relevant_recipes": [
            "Punjabi Vegetable Biryani",
            "Mixed Vegetable Pulao",
            "Vegetable Yakhni Pulao",
            "Spicy Vegetable Rice",
            "Vegetable Masala Pulao",
            "Corn and Peas Pulao",
            "Pumpkin and Lentil Rice",
            "Mixed Lentil Khichdi",
            "Baingan Biryani"
        ]
    },
    {
        "user_input": "Spinach, Lentils, Garlic, Ghee",
        "relevant_recipes": [
            "Palak Khichdi",
            "Spinach and Rice",
            "Saag Pulao",
            "Palak Gosht Chawal",
            "Spinach Biryani",
            "Chickpea and Spinach Pulao",
            "Prawn and Spinach Rice",
            "Palak Shorba Pulao"
        ]
    },
    {
        "user_input": "Rice, Cashews, Raisins, Saffron",
        "relevant_recipes": [
            "Saffron Chicken Pulao",
            "Biryani with Dried Fruits",
            "Chicken and Raisin Rice",
            "Biryani with Raisins",
            "Sweet Rice with Raisins",
            "Sweet Pulao",
            "Rice with Cashews"
        ]
    },
    {
        "user_input": "Chicken, Tikka, Spices, Garlic",
        "relevant_recipes": [
            "Chicken Tikka Pulao",
            "Tikka Biryani",
            "Karachi Tikka Biryani",
            "Chicken Tikka Rice",
            "BBQ Chicken Fried Rice"
        ]
    },
    {
        "user_input": "Coconut, Lemongrass, Fish, Cream",
        "relevant_recipes": [
            "Coconut Fish Pulao",
            "Coconut Prawn Biryani",
            "Thai Green Curry Pulao",
            "Chicken Coconut Rice",
            "Coconut Lemon Rice",
            "Creamy Coconut Rice"
        ]
    },
    {
        "user_input": "Rice, Yogurt, Spices, Vegetables",
        "relevant_recipes": [
            "Vegetable Khichdi",
            "Khichdi with Vegetables",
            "Vegetable Yakhni Pulao",
            "Chana Daal Rice",
            "Chickpea Pulao",
            "Jeera Rice",
            "Bagharay Chawal"
        ]
    },

{
    "user_input": "Whole wheat flour, ghee, spices",
    "relevant_recipes": [
        "Paratha",
        "Aloo Paratha",
        "Puri",
        "Anda Paratha",
        "Methi Thepla",
        "Vegetable Paratha",
        "Paneer Paratha",
        "Egg and Cheese Paratha"
    ]
},
{
    "user_input": "Eggs, spices, onions, tomatoes",
    "relevant_recipes": [
        "Egg Curry",
        "Bhurji",
        "Egg Bhurji",
        "Keema Ghotala",
        "Masala Oats"
    ]
},
{
    "user_input": "Mixed vegetables, spices, flour",
    "relevant_recipes": [
        "Bhaji",
        "Bhaji Puri",
        "Vegetable Sandwich",
        "Vegetable Upma",
        "Thalipeeth"
    ]
},
{
    "user_input": "Milk, sugar, ghee",
    "relevant_recipes": [
        "Pudding",
        "Atta Halwa",
        "Seviyan",
        "Sweets Roti"
    ]
},
{
    "user_input": "Rice, lentils, spices",
    "relevant_recipes": [
        "Idli",
        "Rawa Dosa",
        "Vegetable Upma",
        "Kacha Biryani"
    ]
},
{
    "user_input": "Bread, vegetables, spices",
    "relevant_recipes": [
        "Vegetable Sandwich",
        "Bhaji Sandwich",
        "Paneer Bhurji Sandwich"
    ]
},
{
    "user_input": "Mutton, yogurt, spices, onions",
    "relevant_recipes": [
        "Mutton Korma",
        "Dahi Ghosht"
    ]
},
{
    "user_input": "Semolina, vegetables, spices",
    "relevant_recipes": [
        "Upma",
        "Vegetable Upma"
    ]
},
{
    "user_input": "Pizza base, spices, vegetables",
    "relevant_recipes": [
        "Nihari Pizza",
        "Dahi Puri Pizza",
        "Methi Thepla Quesadilla"
    ]
},
{
    "user_input": "Chickpea flour, spices, yogurt",
    "relevant_recipes": [
        "Dhokla",
        "Methi Thepla"
    ]
},
{
    "user_input": "Fish, spices, onions, tomatoes",
    "relevant_recipes": [
        "Sindhi Fish Curry",
        "Masala Fish",
        "Fish Salan",
        "Fish Tikka Masala",
        "Fish Curry with Spinach",
        "Fish and Vegetable Curry",
        "Fish and Eggplant Curry"
    ]
},
{
    "user_input": "Prawns, spices, onions, tomatoes",
    "relevant_recipes": [
        "Biryani Prawns",
        "Prawn Karahi",
        "Prawn Masala",
        "Prawn Bhuna",
        "Prawn Curry with Potatoes",
        "Prawn and Vegetable Curry"
    ]
},
{
    "user_input": "Fish, yogurt, spices, lemon",
    "relevant_recipes": [
        "Fish Tandoori",
        "Fish Tikka Masala",
        "Fish Korma",
        "Fish Zafrani",
        "Fish and Cilantro Pesto Pasta",
        "Fish Kathi Rolls"
    ]
},
{
    "user_input": "Prawns, yogurt, spices, lemon",
    "relevant_recipes": [
        "Prawn Tandoori",
        "Prawn Korma",
        "Prawn Qorma",
        "Prawn Masala Risotto",
        "Prawn Curry Pot Pie",
        "Prawn Tikka Biryani"
    ]
},
{
    "user_input": "Fish, chickpea flour, spices",
    "relevant_recipes": [
        "Fish Pakora",
        "Fish Pakoras with Mint Chutney",
        "Fish and Corn Fritters",
        "Fish Bonda",
        "Masala Fish Cakes"
    ]
},
{
    "user_input": "Prawns, chickpea flour, spices",
    "relevant_recipes": [
        "Prawn Pakora Spring Rolls",
        "Prawn Fritters",
        "Prawn Samosa",
        "Prawn Samosa Chaat",
        "Prawn and Sweet Corn Fritters"
    ]
},
{
    "user_input": "Fish, rice, spices",
    "relevant_recipes": [
        "Fish Biryani Stuffed Peppers",
        "Fish and Pea Pulao",
        "Spicy Fish Biryani Bowl",
        "Fish Khichdi"
    ]
},
{
    "user_input": "Prawns, rice, spices",
    "relevant_recipes": [
        "Biryani Prawns",
        "Prawn Fried Rice",
        "Prawn and Vegetable Pulao",
        "Prawn Biryani with Coconut",
        "Prawn and Pea Risotto"
    ]
},
{
    "user_input": "Fish, coconut, spices",
    "relevant_recipes": [
        "Malabar Fish Curry",
        "Fish and Coconut Curry",
        "Fish Curry with Mango Salsa",
        "Fish and Mango Quesadilla",
        "Fish and Chaat Masala Fries"
    ]
},
{
    "user_input": "Prawns, coconut, spices",
    "relevant_recipes": [
        "Prawn Curry",
        "Prawn and Mango Curry",
        "Prawn Salad with Coconut Dressing",
        "Prawn Biryani with Coconut"
    ]
},
{
    "user_input": "Fish, bread, spices",
    "relevant_recipes": [
        "Fish Sandwich",
        "Fish Roll",
        "Fish Burger",
        "Fish Tikka Wraps",
        "Fish Burger with Raita"
    ]
},
{
    "user_input": "Prawns, bread, spices",
    "relevant_recipes": [
        "Prawn Samosa",
        "Prawn Tikka Masala Pizza",
        "Prawn Qorma Pasta",
        "Prawn and Egg Roll",
        "Prawn Bruschetta"
    ]
},
{
    "user_input": "Fish, cream, spices",
    "relevant_recipes": [
        "Fish Zafrani",
        "Fish Gratin",
        "Fish and Cheese Stuffed Mushrooms",
        "Fish Curry Burritos"
    ]
},
{
    "user_input": "Prawns, cream, spices",
    "relevant_recipes": [
        "Prawn Masala Dosa",
        "Prawn Fettuccine",
        "Prawn and Sweet Potato Mash",
        "Prawn Masala Risotto"
    ]
},
{
    "user_input": "Minced meat, ginger garlic paste, green chilies, cumin, coriander, salt",
    "relevant_recipes": [
        "Spicy Minced Meat Curry",
        "Ginger Garlic Minced Kebabs",
        "Green Chili Minced Meat Tacos",
        "Coriander Minced Meat Stew"
    ]
},
{
    "user_input": "Ground beef, tomatoes, green chilies, coriander, cumin, pomegranate seeds",
    "relevant_recipes": [
        "Tomato and Ground Beef Stew",
        "Pomegranate Beef Chili",
        "Green Chili Beef Enchiladas",
        "Cumin Beef Tomato Soup"
    ]
},
{
    "user_input": "Beef strips, yogurt, ginger garlic paste, fried onions, mustard oil",
    "relevant_recipes": [
        "Beef Yogurt Curry",
        "Fried Onion Beef Stir-Fry",
        "Mustard Oil Beef Skewers",
        "Ginger Garlic Beef Strips"
    ]
},
{
    "user_input": "Chicken cubes, cream, yogurt, ginger garlic paste, green chilies",
    "relevant_recipes": [
        "Creamy Chicken Curry",
        "Green Chili Chicken Masala",
        "Yogurt Chicken Skewers",
        "Ginger Garlic Chicken Stew"
    ]
},
{
    "user_input": "Fish fillets, lemon juice, ginger garlic paste, red chili powder, cumin",
    "relevant_recipes": [
        "Spicy Lemon Fish Curry",
        "Red Chili Fish Tikka",
        "Cumin Fish Stir-Fry",
        "Ginger Garlic Lemon Fish Bake"
    ]
},
    {
        "user_input": "Mutton, basmati rice, yogurt, spices",
        "relevant_recipes": [
            "Meat Biryani",
            "Meat Pulao",
            "Meat Kunna Pulao",
            "Meat Bhuna Biryani",
            "Meat Karahi Biryani",
            "Meat Hyderabadi Biryani"
        ]
    },
    {
        "user_input": "Minced beef, spices, onions, eggs",
        "relevant_recipes": [
            "Beef Shami Kebab",
            "Beef Nargisi Kofta",
            "Beef Anda Curry",
            "Beef Bihari Kebab",
            "Beef Methi Paratha"
        ]
    },
    {
        "user_input": "Beef, spinach, garlic, onions, spices",
        "relevant_recipes": [
            "Beef Palak Gosht",
            "Beef Saag",
            "Beef Palak Khichdi",
            "Beef Palak Daal"
        ]
    },
    {
        "user_input": "Mutton, yogurt, onions, spices",
        "relevant_recipes": [
            "Meat Rogan Josh",
            "Meat Handi",
            "Meat Achar Gosht",
            "Meat Shahi Korma",
            "Meat Methi Gosht"
        ]
    },
    {
        "user_input": "Beef, basmati rice, yogurt, spices",
        "relevant_recipes": [
            "Beef Qorma Biryani",
            "Beef Zafran Biryani",
            "Beef Dum Biryani",
            "Beef Seekh Biryani",
            "Beef Biryani Platter"
        ]
    },
    {
        "user_input": "Beef, tomatoes, onions, green chilies",
        "relevant_recipes": [
            "Beef Karahi Gosht",
            "Beef Peshawari Karahi",
            "Beef Tamatar Gosht",
            "Beef Lahori Karahi",
            "Beef Balti"
        ]
    },
    {
        "user_input": "Mutton, lentils, garlic, ginger, spices",
        "relevant_recipes": [
            "Meat Daal Gosht",
            "Meat Masoor Daal",
            "Meat Chana Daal",
            "Meat Saag Mutton"
        ]
    },
    {
        "user_input": "Beef, okra, onions, tomatoes",
        "relevant_recipes": [
            "Beef Bhindi Gosht",
            "Beef Bhindi Masala",
            "Beef Dahi Bhindi"
        ]
    },
    {
        "user_input": "Beef, potatoes, spices, tomatoes",
        "relevant_recipes": [
            "Beef Aloo Gosht",
            "Beef Aloo Masala",
            "Beef Aloo Tikki"
        ]
    },
    {
        "user_input": "Beef, wheat flour, ginger, garlic",
        "relevant_recipes": [
            "Beef Kunna",
            "Beef Kunna Nihari",
            "Beef Nihari Korma"
        ]
    },

    {
        "user_input": "Beef, Spices, Rice",
        "relevant_recipes": [
            "Beef Biryani",
            "Beef Nihari Biryani",
            "Beef Biryani Wraps",
            "Beef Biryani Lasagna",
            "Beef Biryani Spring Rolls",
            "Beef Biryani Omelette",
            "Beef Biryani Frittata",
            "Beef Biryani Stuffed Cabbage Rolls"
        ]
    },
    {
        "user_input": "Beef, Spices, Noodles",
        "relevant_recipes": [
            "Beef Ramen",
            "Beef Stir-Fried Noodles",
            "Beef Curry Ramen Noodles",
            "Beef Masala Ramen",
            "Beef Tikka Masala Ramen",
            "Beef Jalfrezi Stir-Fried Noodles"
        ]
    },
    {
        "user_input": "Beef, Cheese, Tortillas",
        "relevant_recipes": [
            "Beef Tikka Quesadilla",
            "Beef & Cheese Quesadillas",
            "Beef Vindaloo Quesadillas",
            "Beef Tikka Quesadillas",
            "Beef Tikka Tacos"
        ]
    },
    {
        "user_input": "Beef, Cheese, Vegetables",
        "relevant_recipes": [
            "Beef & Cheese Stuffed Bell Peppers",
            "Beef & Cheese Spring Rolls",
            "Beef & Spinach Stuffed Shells",
            "Beef & Cheese Stuffed Potatoes",
            "Beef & Spinach Pizza"
        ]
    },
    {
        "user_input": "Beef, Spices, Vegetables",
        "relevant_recipes": [
            "Beef & Vegetable Stir-Fry",
            "Beef & Zucchini Stir-Fry",
            "Beef and Zucchini Stir-Fry",
            "Beef & Vegetable Stir-Fry Tacos",
            "Beef and Cauliflower Curry",
            "Beef Curry Stuffed Mushrooms"
        ]
    },
    {
        "user_input": "Beef, Spices, Yogurt",
        "relevant_recipes": [
            "Beef Tandoori Burgers",
            "Beef Korma Enchiladas",
            "Beef Korma Sushi Rolls",
            "Beef Korma Samosas",
            "Beef Korma Tikka Masala",
            "Beef Korma Stuffed Peppers"
        ]
    },
    {
        "user_input": "Beef, Spices, Bread",
        "relevant_recipes": [
            "Beef Meatball Sub",
            "Beef Nihari Sliders",
            "Beef Tandoori Pita",
            "Beef Kofta Sliders",
            "Beef Kebab Sliders",
            "Beef Masala Naan Pockets"
        ]
    },
    {
        "user_input": "Beef, Spices, Potatoes",
        "relevant_recipes": [
            "Beef and Potato Hash",
            "Beef and Potato Casserole",
            "Beef Nihari Stuffed Potatoes",
            "Beef Masala Croquettes",
            "Beef Korma and Couscous",
            "Beef and Sweet Potato Mash"
        ]
    },
    {
        "user_input": "Beef, Spices, Eggs",
        "relevant_recipes": [
            "Beef & Spinach Omelette",
            "Beef Biryani Omelette",
            "Beef Tandoori Frittata",
            "Beef Biryani Frittata",
            "Beef Tikka Masala Quiche"
        ]
    },
    {
        "user_input": "Beef, Spices, Pasta",
        "relevant_recipes": [
            "Beef Kofta Curry Pasta",
            "Beef Tikka Masala Pasta",
            "Beef and Spinach Cannelloni",
            "Beef Masala and Gnocchi Bake",
            "Beef Korma Pesto Pasta",
            "Beef Jalfrezi Risotto"
        ]
    },
    {
        "user_input": "Beef, Spices, Salad",
        "relevant_recipes": [
            "Beef & Mango Salad",
            "Beef Tandoori Caesar Salad",
            "Beef Fajita Salad",
            "Beef and Avocado Toast",
            "Beef Kofta and Couscous Salad"
        ]
    },
    {
        "user_input": "Beef, Spices, Wraps",
        "relevant_recipes": [
            "Beef Shawarma Wrap",
            "Beef Biryani Wraps",
            "Beef Kofta Sliders",
            "Beef Tandoori Naan Wrap",
            "Beef Seekh Kebab Tacos"
        ]
    },
    {
        "user_input": "Beef, Spices, Rice and Beans",
        "relevant_recipes": [
            "Beef Curry Burrito",
            "Beef Masala Burrito"
        ]
    },
    {
        "user_input": "Beef, Spices, Rice and Vegetables",
        "relevant_recipes": [
            "Beef Kofta and Rice Casserole",
            "Beef Curry & Quinoa Bowl",
            "Beef Kofta Chili",
            "Beef Kofta Sliders"
        ]
    },


{
    "user_input": "Beef, Spices, Rice",
    "relevant_recipes": [
        "Beef Biryani Taco Salad",
        "Beef Biryani Gnocchi",
        "Beef and Peas Fried Rice",
        "Beef and Broccoli Stir-Fry",
        "Beef Biryani Gyoza",
        "Beef Biryani Stir-Fry",
        "Beef and Vegetable Stir-Fried Rice",
        "Beef Biryani Sliders",
        "Beef Nihari and Couscous",
        "Beef Nihari Gnocchi"
    ]
},
{
    "user_input": "Beef, Spices, Potatoes",
    "relevant_recipes": [
        "Beef and Potato Curry",
        "Beef Kofta Shepherd’s Pie",
        "Beef and Cheese Stuffed Peppers",
        "Beef and Potato Samosas",
        "Beef Potato Casserole",
        "Beef Nihari Goulash"
    ]
},
{
    "user_input": "Beef, Vegetables, Soy Sauce",
    "relevant_recipes": [
        "Beef and Broccoli Stir-Fry",
        "Beef and Vegetable Pasta",
        "Beef and Vegetable Korma",
        "Beef and Vegetable Stir-Fried Rice",
        "Beef and Mushroom Stroganoff"
    ]
},
{
    "user_input": "Beef, Cheese, Spices",
    "relevant_recipes": [
        "Beef Korma Stuffed Bell Peppers",
        "Beef and Cheese Stuffed Naan",
        "Beef and Cheese Stuffed Peppers",
        "Beef and Cheese Quesadillas",
        "Beef Tikka Stuffed Bell Peppers",
        "Beef Masala Tacos"
    ]
},
{
    "user_input": "Beef, Rice, Broccoli",
    "relevant_recipes": [
        "Beef and Broccoli Curry",
        "Beef and Broccoli Quesadillas",
        "Beef and Broccoli Stir-Fry"
    ]
},
{
    "user_input": "Beef, Noodles, Spices",
    "relevant_recipes": [
        "Beef Korma Stir-Fried Noodles",
        "Beef Tikka Ramen",
        "Beef Seekh Kebab Pasta",
        "Beef Korma Stir-Fried Noodles"
    ]
},
{
    "user_input": "Beef, Spinach, Spices",
    "relevant_recipes": [
        "Beef and Spinach Curry",
        "Beef Palak Paneer",
        "Beef and Spinach Pasta"
    ]
},

{
    "user_input": "Rice, Mixed Vegetables, Spices",
    "relevant_recipes": [
        "Aloo Pulao",
        "Masoor Pulao",
        "Vegetable Pulao with Raita",
        "Baked Vegetable Rice",
        "Punjabi Methi Pulao",
        "Peas and Carrot Rice",
        "Jalfrezi Rice",
        "Spinach and Mushroom Rice",
        "Basmati Rice with Lentils",
        "Spinach and Potato Rice",
        "Spiced Cabbage Rice",
        "Lentil and Vegetable Biryani",
        "Vegetable Risotto",
        "Peas Pulao",
        "Vegetable Samosa Rice",
        "Methi and Rice",
        "Khichdi with Spinach",
        "Pulao with Nuts",
        "Masoor and Vegetable Pulao",
        "Green Chili Rice",
        "Spicy Lentil Rice",
        "Spiced Pumpkin Rice",
        "Dal Makhani Rice",
        "Raita and Rice",
        "Vegetable Tikka Rice",
        "Tandoori Veg Biryani",
        "Vegetable Quinoa Pulao",
        "Green Peas and Mint Rice",
        "Mixed Vegetable Fried Rice",
        "Vegetable Pad Thai Rice",
        "Spiced Chickpea and Spinach Rice",
        "Lauki Pulao",
        "Kadhi Chawal",
        "Vegetable Szechuan Rice",
        "Vegetable Seekh Kebabs Rice",
        "Pindi Chana Rice",
        "Vegetable Biryani with Raita",
        "Vegetable Biryani with Yogurt",
        "Vegetable Pulao with Raisins",
        "Pumpkin and Pea Pulao",
        "Vegetable Porridge",
        "Stir-Fried Vegetable Rice",
        "Vegetable Qorma Rice",
        "Kadai Vegetable Rice",
        "Sweet Corn and Pea Rice",
        "Vegetable Frittata Rice"
    ]
},
{
    "user_input": "Basmati Rice, Spices",
    "relevant_recipes": [
        "Basmati Rice with Lentils",
        "Spinach and Potato Rice",
        "Spiced Cabbage Rice",
        "Peas Pulao",
        "Methi and Rice",
        "Green Chili Rice",
        "Lemon and Coriander Rice",
        "Spicy Lentil Rice",
        "Saffron and Nut Rice",
        "Ghee Rice",
        "Masala Dosa Rice",
        "Rawa Idli Rice",
        "Vegetable Taco Rice",
        "Gajar Ka Halwa Rice",
        "Vegetable Paella",
        "Gobi Manchurian Rice",
        "Spiced Chickpea and Spinach Rice",
        "Kadhi Chawal",
        "Pista Pulao",
        "Methi Khichdi",
        "Pulao with Nuts",
        "Vegetable Biryani with Yogurt",
        "Sweet Rice",
        "Vegetable Bhel Rice",
        "Tandoori Vegetable Pulao",
        "Rawa Rice",
        "Punjabi Pulao",
        "Tandoori Veg Biryani",
        "Ghee Rice",
        "Masoor Dal Pulao",
        "Palak Chole Rice",
        "Sweet Potato Rice",
        "Herb Rice",
        "Thai Basil Rice"
    ]
},
{
    "user_input": "Basmati Rice, Mixed Vegetables, Spices",
    "relevant_recipes": [
        "Aloo Pulao",
        "Vegetable Pulao with Raita",
        "Peas and Carrot Rice",
        "Spiced Cabbage Rice",
        "Lentil and Vegetable Biryani",
        "Vegetable Risotto",
        "Masoor and Vegetable Pulao",
        "Vegetable Quinoa Pulao",
        "Vegetable Paella",
        "Pulao with Nuts",
        "Green Peas and Mint Rice",
        "Mixed Vegetable Fried Rice",
        "Vegetable Pad Thai Rice",
        "Spiced Chickpea and Spinach Rice",
        "Kadhi Chawal",
        "Vegetable Seekh Kebabs Rice",
        "Vegetable Biryani with Raita",
        "Vegetable Biryani with Yogurt",
        "Vegetable Pulao with Raisins",
        "Pumpkin and Pea Pulao",
        "Vegetable Porridge",
        "Stir-Fried Vegetable Rice",
        "Vegetable Qorma Rice",
        "Kadai Vegetable Rice",
        "Sweet Corn and Pea Rice",
        "Vegetable Frittata Rice",
        "Pind Chana Rice"
    ]
},
{
    "user_input": "Rice, Lemon, Garlic, Spices",
    "relevant_recipes": [
        "Lemon Garlic Rice",
        "Zesty Rice"
    ]
},
{
    "user_input": "Basmati Rice, Spinach, Spices",
    "relevant_recipes": [
        "Spinach and Potato Rice",
        "Spinach and Cheese Biryani",
        "Spinach and Cheese Casserole Rice",
        "Spinach and Mushroom Rice",
        "Spinach and Tomato Rice",
        "Biryani with Spinach",
        "Palak Chole Rice",
        "Khichdi with Spinach",
        "Spiced Chickpea and Spinach Rice"
    ]
},
{
    "user_input": "Basmati Rice, Mango, Coconut, Spices",
    "relevant_recipes": [
        "Mango and Coconut Rice"
    ]
},

{
    "user_input": "Chickpeas, potatoes, spices, yogurt",
    "relevant_recipes": [
        "Chole Tikki",
        "Puri Chaat",
        "Aloo Papri Chaat",
        "Pani Puri Chat",
        "Chana Puri",
        "Pani Puri Chaat",
        "Dahi Wale Aloo",
        "Chaat Masala Fries",
        "Chana Masala Tacos",
        "Chana Chaat Pizza",
        "Chaat Masala Nachos",
        "Chaat Roti",
        "Chana Daal Puri",
        "Chaat Gali"
    ]
},
{
    "user_input": "Chicken, spices, yogurt",
    "relevant_recipes": [
        "Tandoori Chicken Tikka",
        "Tikka Boti",
        "Chicken Biryani Samosa",
        "Chicken Tikka Poutine",
        "Chicken Seekh Kebab Nachos",
        "Tandoori Chicken Bites",
        "Chicken Tikka Salad",
        "Fish Tikka Wrap",
        "Tandoori Chicken Tacos",
        "Tandoori Chaat",
        "Chicken Karahi Sliders",
        "Chicken Tikka Fries"
    ]
},
{
    "user_input": "Rice, spices, meat",
    "relevant_recipes": [
        "Biryani Pulao",
        "Biryani Chaat",
        "Biryani Tacos",
        "Vegetable Manchurian Biryani",
        "Biryani Pockets",
        "Chicken Biryani Sushi",
        "Biryani Roll",
        "Biryani Quesadilla",
        "Biryani Burrito",
        "Biryani Chaat",
        "Prawn Biryani Spring Rolls"
    ]
},
{
    "user_input": "Potatoes, spices, chutney",
    "relevant_recipes": [
        "Puri Chaat",
        "Aloo Papri Chaat",
        "Chaat Masala Fries",
        "Pani Puri Chat",
        "Chaat Masala Nachos",
        "Tikki Chaat",
        "Bhaji (Vegetable Fritters)",
        "Bhaji (Fried Vegetables)",
        "Pani Puri Shots",
        "Masala Mac and Cheese",
        "Spicy Chicken Fries"
    ]
},
{
    "user_input": "Paneer, spices, flatbread",
    "relevant_recipes": [
        "Paneer Tikka Roll",
        "Paneer Tikka Fries",
        "Paneer Tikka Burger",
        "Paneer Masala Tacos",
        "Paneer Tikka Quesadilla",
        "Pesto Paneer Tikka",
        "Tandoori Paneer Fries"
    ]
},


    {
        "user_input": "Gram flour, sugar, ghee, cardamom, pistachios",
        "relevant_recipes": [
            "Besan ka Ladoo",
            "Besan Patisa",
            "Patisa",
            "Sohan Papdi",
            "Boondi Ladoo"
        ]
    },
    {
        "user_input": "Milk, sugar, cardamom, almonds",
        "relevant_recipes": [
            "Seviyan Kheer",
            "Badam Ka Halwa",
            "Phirni",
            "Gajar ka Halwa",
            "Khoya Barfi",
            "Ladoo Pethiyan"
        ]
    },
    {
        "user_input": "Milk, sugar, cardamom, saffron",
        "relevant_recipes": [
            "Shahi Tukda",
            "Chawal ki Kheer",
            "Jalebi Shahi Tukda",
            "Saffron Phirni"
        ]
    },
    {
        "user_input": "Coconut, sugar, milk, cardamom",
        "relevant_recipes": [
            "Coconut Barfi",
            "Coconut Ladoo",
            "Coconut Phirni"
        ]
    },
    {
        "user_input": "Rice, milk, sugar, cardamom",
        "relevant_recipes": [
            "Chawal ki Kheer",
            "Doodh Pak",
            "Makhana Kheer",
            "Rice flour Phirni"
        ]
    },
    {
        "user_input": "Pumpkin, sugar, ghee, cardamom",
        "relevant_recipes": [
            "Kaddu ka Halwa",
            "Kashi Halwa",
            "Pumpkin Halwa"
        ]
    },
    {
        "user_input": "Flour, sugar, ghee, cardamom",
        "relevant_recipes": [
            "Bakarkhani",
            "Gulab Jamun Cake",
            "Riwari",
            "Shakarpariyan"
        ]
    },
    {
        "user_input": "Almonds, sugar, ghee, cardamom",
        "relevant_recipes": [
            "Badam Pista Kheer",
            "Badam Ka Halwa",
            "Thadal",
            "Gajar Barfi",
            "Dal Badam Halwa"
        ]
    },
    {
        "user_input": "Rice flour, milk, sugar, cardamom",
        "relevant_recipes": [
            "Phirni",
            "Saffron Phirni",
            "Ghulab Phirni",
            "Boondi Phirni"
        ]
    },
    {
        "user_input": "Mango pulp, milk, sugar, cardamom",
        "relevant_recipes": [
            "Mango Kulfi",
            "Mango Rabri"
        ]
    },
    {
        "user_input": "Gram flour, sugar, ghee, cardamom, almonds",
        "relevant_recipes": [
            "Besan ka Ladoo",
            "Ladoo Pethiyan",
            "Gajar Barfi"
        ]
    },


{
  "user_input": "Coconut, Sugar, Cardamom",
  "relevant_recipes": [
    "Lapsi",
    "Lauki Halwa",
    "Makhana Halwa",
    "Suji Badam Halwa",
    "Besan ka Halwa",
    "Pistachio Coconut Ice Cream",
    "Coconut Mango Mousse",
    "Coconut Mango Parfait",
    "Coconut Mango Cake",
    "Coconut & Mango Cheesecake",
    "Coconut & Date Rolls",
    "Coconut Chia Energy Bars",
    "Coconut & Almond Rice Balls",
    "Coconut & Pistachio Muffins",
    "Coconut Chia Seed Pudding",
    "Coconut & Mango Cake"
  ]
},
{
  "user_input": "Milk, Sugar, Cardamom",
  "relevant_recipes": [
    "Pistachio Kulfi",
    "Banana Kheer",
    "Shahi Tukra",
    "Moong Dal Ladoo",
    "Suji Badam Halwa",
    "Malai Custard Eclairs",
    "Cardamom Saffron Donuts",
    "Chocolate Chip Kulfi",
    "Mango Kulfi Cheesecake",
    "Saffron Almond Blondies",
    "Rose Gulab Jamun Brownies",
    "Kheer Pancakes",
    "Saffron Rice Pudding",
    "Ginger & Cardamom Cookies",
    "Saffron & Cardamom Biscotti"
  ]
},
{
  "user_input": "Pistachios, Sugar, Cardamom",
  "relevant_recipes": [
    "Pistachio Kulfi",
    "Kesar Pista Barfi",
    "Pistachio & Rose Baklava Rolls",
    "Rose & Pistachio Macarons",
    "Pistachio Coconut Ice Cream",
    "Rose & Pistachio Rice Pudding",
    "Pistachio & Date Roll",
    "Honey Pistachio Baklava Cups"
  ]
},
{
  "user_input": "Flour, Sugar, Butter",
  "relevant_recipes": [
    "Jalebi Rabri",
    "Shakarpare",
    "Zafrani Bread Pudding",
    "Cinnamon Saffron Rolls",
    "Rose & Pistachio Macarons",
    "Cardamom Saffron Donuts",
    "Nutty Saffron Cookies",
    "Chocolate Coconut Pudding",
    "Lemon Rose Cupcakes",
    "Ginger & Honey Pudding",
    "Chocolate Chai Brownies",
    "Saffron & Cardamom Biscotti"
  ]
},
{
  "user_input": "Mango, Coconut, Sugar",
  "relevant_recipes": [
    "Mango Coconut Mousse",
    "Mango Coconut Barfi",
    "Mango Coconut Parfait",
    "Mango & Coconut Cheesecake",
    "Coconut Mango Popsicles",
    "Mango & Coconut Cream Tarts",
    "Mango & Coconut Cake"
  ]
},
{
  "user_input": "Dates, Coconut, Cardamom",
  "relevant_recipes": [
    "Date & Almond Energy Bites",
    "Coconut & Date Rolls",
    "Pistachio & Date Roll"
  ]
},
{
  "user_input": "Rice, Sugar, Cardamom",
  "relevant_recipes": [
    "Rose & Pistachio Rice Pudding",
    "Saffron Rice Pudding",
    "Coconut & Almond Rice Balls"
  ]
},
{
  "user_input": "Chia Seeds, Coconut, Sugar",
  "relevant_recipes": [
    "Coconut Chia Halwa",
    "Saffron Chia Pudding",
    "Coconut Chia Seed Pudding",
    "Coconut Chia Energy Bars"
  ]
},

{
    "user_input": "Coconut, Mango, Sugar, Milk",
    "relevant_recipes": [
        "Mango Coconut Smoothie",
        "Mango Coconut Trifle",
        "Mango & Coconut Lassi",
        "Mango & Coconut Mousse",
        "Mango Coconut Popsicles",
        "Mango Pudding with Coconut Cream",
        "Mango Coconut Scones"
    ]
},
{
    "user_input": "Coconut, Dates, Sugar",
    "relevant_recipes": [
        "Date & Coconut Energy Bites",
        "Date & Coconut Halwa",
        "Coconut & Date Bites",
        "Coconut Date Fudge"
    ]
},
{
    "user_input": "Chocolate, Chai Spices, Sugar, Flour",
    "relevant_recipes": [
        "Spiced Chocolate Bark",
        "Spiced Chocolate Cake",
        "Spiced Chocolate Muffins",
        "Chocolate Chai Truffles",
        "Spiced Chocolate Brownies",
        "Chai-Spiced Cupcakes"
    ]
},
{
    "user_input": "Coconut, Almonds, Sugar",
    "relevant_recipes": [
        "Almond & Coconut Fudge",
        "Coconut Almond Ladoo",
        "Almond Joy Cheesecake",
        "Pistachio Coconut Energy Balls",
        "Coconut Almond Cake"
    ]
},
{
    "user_input": "Coconut, Mango, Honey",
    "relevant_recipes": [
        "Mango & Coconut Mousse",
        "Mango Coconut Smoothie",
        "Mango Chia Seed Pudding",
        "Mango & Coconut Trifle",
        "Mango Coconut Cheesecake"
    ]
},
{
    "user_input": "Coconut, Sugar, Egg, Flour",
    "relevant_recipes": [
        "Coconut Flour Brownies",
        "Coconut Macaroon Cake",
        "Coconut Saffron Rice Pudding",
        "Coconut Lime Bars"
    ]
},


    {
        "user_input": "Coconut, Mango, Sugar",
        "relevant_recipes": [
            "Mango Coconut Parfait",
            "Mango Coconut Muffins",
            "Mango Coconut Pie",
            "Mango Coconut Ice Cream",
            "Mango Coconut Popsicles",
            "Mango Coconut Fritters",
            "Mango Coconut Mousse",
            "Mango Coconut Granola Bars",
            "Mango Coconut Tiramisu",
            "Mango Coconut Rice Cake"
        ]
    },
    {
        "user_input": "Chocolate, Coconut, Sugar",
        "relevant_recipes": [
            "Chocolate Coconut Cheesecake",
            "Chocolate Coconut Bars",
            "Chocolate Coconut Chia Seed Pudding",
            "Chocolate Coconut Trifle",
            "Chocolate Coconut Fudge",
            "Chocolate Almond Biscotti",
            "Chocolate Almond Cake",
            "Chocolate Chai Fudge",
            "Chocolate Cardamom Pudding",
            "Chocolate Almond Bark"
        ]
    },
    {
        "user_input": "Dates, Almonds, Coconut",
        "relevant_recipes": [
            "Date & Coconut Fudge",
            "Chocolate Covered Dates",
            "Coconut Date Squares",
            "Date & Chocolate Smoothie",
            "Date & Almond Energy Bars",
            "Coconut Almond Fudge",
            "Coconut Almond Crumble",
            "Coconut Almond Rice Bars"
        ]
    },
    {
        "user_input": "Mango, Coconut, Flour",
        "relevant_recipes": [
            "Mango Coconut Muffins",
            "Mango Coconut Rice Cake",
            "Mango Coconut Pie",
            "Mango Coconut Fritters",
            "Mango Coconut Tart",
            "Mango Coconut Popsicles",
            "Mango Coconut Pancakes"
        ]
    },
    {
        "user_input": "Chia Seeds, Almond Milk, Honey",
        "relevant_recipes": [
            "Chocolate Mint Chia Pudding",
            "Mango Chia Seed Parfait",
            "Chocolate Chia Seed Bars",
            "Coconut Chia Seed Bars",
            "Chocolate Coconut Chia Seed Pudding"
        ]
    },
    {
        "user_input": "Mango, Lime, Sugar",
        "relevant_recipes": [
            "Mango & Lime Sorbet",
            "Mango Coconut Popsicles",
            "Mango Coconut Pie",
            "Mango & Coconut Tart"
        ]
    },
    {
        "user_input": "Coconut, Cardamom, Sugar",
        "relevant_recipes": [
            "Almond Cardamom Cookies",
            "Coconut Cardamom Cake",
            "Chai-Spiced Coconut Cake",
            "Chai-Spiced Coconut Scones",
            "Coconut Mango Rice Cake"
        ]
    },
    {
        "user_input": "Mango, Coconut, Milk",
        "relevant_recipes": [
            "Mango Coconut Mousse",
            "Mango Coconut Cream Bars",
            "Mango Coconut Custard",
            "Mango Coconut Pie",
            "Mango Coconut Ice Cream",
            "Mango Coconut Rice Cake"
        ]
    },
    {
        "user_input": "Chocolate, Dates, Cocoa Powder",
        "relevant_recipes": [
            "Date & Chocolate Smoothie",
            "Date & Cocoa Bliss Balls",
            "Chocolate Chai Fudge",
            "Chocolate Almond Biscotti"
        ]
    },
    {
        "user_input": "Almond Flour, Sugar, Shredded Coconut",
        "relevant_recipes": [
            "Almond Coconut Cookies",
            "Coconut Almond Crumble",
            "Almond Coconut Granola",
            "Almond Coconut Rice Bars",
            "Coconut Almond Pudding",
            "Chocolate Coconut Bars"
        ]
    },
    {
        "user_input": "Chocolate, Almonds, Sugar",
        "relevant_recipes": [
            "Chocolate Almond Bark",
            "Chocolate Almond Cake",
            "Chocolate Almond Biscotti",
            "Chocolate Chai Fudge"
        ]
    },
    {
        "user_input": "Coconut, Pineapple, Sugar",
        "relevant_recipes": [
            "Coconut Pineapple Cake",
            "Coconut Pineapple Parfait",
            "Coconut Mango Salad"
        ]
    },
    {
        "user_input": "Chai Spices, Coconut, Sugar",
        "relevant_recipes": [
            "Chocolate Chai Ice Cream",
            "Chai-Spiced Coconut Macarons",
            "Chai-Spiced Coconut Pudding",
            "Chai-Spiced Coconut Cake",
            "Chai-Spiced Coconut Scones",
            "Chai-Spiced Pineapple Cake"
        ]
    },
    {
        "user_input": "Mango, Banana, Coconut",
        "relevant_recipes": [
            "Mango Almond Smoothie",
            "Coconut Mango Whip",
            "Mango Coconut Bliss Balls"
        ]
    },
    {
        "user_input": "Chocolate, Mango, Whipped Cream",
        "relevant_recipes": [
            "Chocolate Mango Parfait",
            "Chocolate Mango Tiramisu"
        ]
    },
    {
        "user_input": "Mango, Dates, Shredded Coconut",
        "relevant_recipes": [
            "Mango Coconut Bliss Balls",
            "Date & Coconut Fudge",
            "Mango Coconut Energy Bites"
        ]
    },
    {
        "user_input": "Mango, Almond Flour, Honey",
        "relevant_recipes": [
            "Mango Almond Rice Pudding",
            "Mango Almond Popsicles",
            "Mango Almond Energy Bites"
        ]
    },

    {
        "user_input": "Rooh Afza syrup, water, lemon juice, ice",
        "relevant_recipes": [
            "Rooh Afza",
            "Rooh Afza Mojito"
        ]
    },
    {
        "user_input": "Raw mango, sugar, black salt, mint, water",
        "relevant_recipes": [
            "Aam Panna",
            "Aam Sharbat"
        ]
    },
    {
        "user_input": "Lemons, water, sugar, mint",
        "relevant_recipes": [
            "Lemonade",
            "Shikanji",
            "Mint Lemon Cooler",
            "Strawberry Lemonade",
            "Falsa Lemonade",
            "Lemon Fizz",
            "Lemonade Iced Tea"
        ]
    },
    {
        "user_input": "Yogurt, water, cumin, salt, mint",
        "relevant_recipes": [
            "Chaas",
            "Spiced Buttermilk"
        ]
    },
    {
        "user_input": "Fresh mint, lemon juice, soda, sugar",
        "relevant_recipes": [
            "Mint Margarita",
            "Mint Lemon Cooler"
        ]
    },
    {
        "user_input": "Jaljeera powder, water, lemon juice, mint",
        "relevant_recipes": [
            "Jaljeera"
        ]
    },
    {
        "user_input": "Fresh coconut",
        "relevant_recipes": [
            "Coconut Water",
            "Coconut Lemonade",
            "Coconut Milkshake",
            "Coconut Mojito"
        ]
    },
    {
        "user_input": "Pomegranate, sugar, lemon",
        "relevant_recipes": [
            "Pomegranate Juice"
        ]
    },
    {
        "user_input": "Rose petals, sugar, water, lemon juice",
        "relevant_recipes": [
            "Gulaab Sharbat"
        ]
    },
    {
        "user_input": "Ripe mango, sugar, water, mint",
        "relevant_recipes": [
            "Aam Sharbat",
            "Mango Mint Lassi"
        ]
    },
    {
        "user_input": "Fennel seeds, sugar, water",
        "relevant_recipes": [
            "Saunf Ka Sharbat"
        ]
    },
    {
        "user_input": "Khus khus (vetiver), sugar, water",
        "relevant_recipes": [
            "Khus Khus Sharbat"
        ]
    },
    {
        "user_input": "Almonds, milk, sugar, cardamom, saffron",
        "relevant_recipes": [
            "Badam Milk"
        ]
    },
    {
        "user_input": "Coffee, milk, sugar, ice",
        "relevant_recipes": [
            "Chilled Coffee",
            "Cold Coffee",
            "Cardamom Coffee Shake"
        ]
    },
    {
        "user_input": "Milk, almonds, pistachios, fennel seeds, sugar, cardamom",
        "relevant_recipes": [
            "Thandai"
        ]
    },
    {
        "user_input": "Chia seeds, water, lemon, honey",
        "relevant_recipes": [
            "Chia Seed Drink"
        ]
    },
    {
        "user_input": "Strawberries, milk, sugar",
        "relevant_recipes": [
            "Strawberry Shake"
        ]
    },
    {
        "user_input": "Milk, saffron, sugar",
        "relevant_recipes": [
            "Kesar Milk"
        ]
    },
    {
        "user_input": "Oranges",
        "relevant_recipes": [
            "Orange Juice"
        ]
    },
    {
        "user_input": "Tamarind, sugar, water",
        "relevant_recipes": [
            "Tamarind Drink",
            "Tamarind Fizz",
            "Spicy Tamarind Mocktail"
        ]
    },
    {
        "user_input": "Pineapple",
        "relevant_recipes": [
            "Pineapple Juice",
            "Pineapple Ginger Drink"
        ]
    },
    {
        "user_input": "Lychee",
        "relevant_recipes": [
            "Lychee Juice",
            "Lychee Mojito",
            "Lychee Mint Lemonade"
        ]
    },
    {
        "user_input": "Ginger, sugar, water",
        "relevant_recipes": [
            "Ginger Ale",
            "Ginger Lemon Fizz",
            "Pineapple Ginger Drink"
        ]
    },
    {
        "user_input": "Watermelon",
        "relevant_recipes": [
            "Watermelon Juice",
            "Watermelon Slush"
        ]
    },
    {
        "user_input": "Falsa berries, sugar, water",
        "relevant_recipes": [
            "Falsa Drink",
            "Falsa Lemonade"
        ]
    },
    {
        "user_input": "Almonds, sugar, milk, saffron",
        "relevant_recipes": [
            "Badaam Ka Sharbat"
        ]
    },
    {
        "user_input": "Dates, milk, sugar",
        "relevant_recipes": [
            "Khajoor Shake",
            "Date Shake"
        ]
    },
    {
        "user_input": "Yogurt, water, salt",
        "relevant_recipes": [
            "Buttermilk",
            "Spiced Buttermilk"
        ]
    },
    {
        "user_input": "Melon, sugar, water",
        "relevant_recipes": [
            "Kharboza Drink"
        ]
    },
    {
        "user_input": "Grapes",
        "relevant_recipes": [
            "Grape Juice"
        ]
    },
    {
        "user_input": "Fennel seeds, water",
        "relevant_recipes": [
            "Fennel Seed Water"
        ]
    },
    {
        "user_input": "Ginseng, honey, water",
        "relevant_recipes": [
            "Ginseng Tea"
        ]
    },
    {
        "user_input": "Various herbs, water",
        "relevant_recipes": [
            "Herbal Tea"
        ]
    },
    {
        "user_input": "Black currants, sugar, water",
        "relevant_recipes": [
            "Black Currant Juice"
        ]
    },
    {
        "user_input": "Milk, pistachios, sugar",
        "relevant_recipes": [
            "Pista Milk"
        ]
    },
    {
        "user_input": "Lemon juice, soda, sugar",
        "relevant_recipes": [
            "Lemon Fizz",
            "Mint Lemon Cooler"
        ]
    },
    {
        "user_input": "Black tea, spices, milk, sugar",
        "relevant_recipes": [
            "Masala Chai",
            "Khus Khus Chai",
            "Masala Chai Frappe"
        ]
    },
    {
        "user_input": "Coconut water, lemon juice, sugar",
        "relevant_recipes": [
            "Coconut Lemonade"
        ]
    },
    {
        "user_input": "Lemons, sugar, salt, water",
        "relevant_recipes": [
            "Nimbu Pani"
        ]
    },
    {
        "user_input": "Milk, rose syrup, sugar",
        "relevant_recipes": [
            "Rose Milk"
        ]
    },
    {
        "user_input": "Chaunsa mango, sugar, water",
        "relevant_recipes": [
            "Chaunsa Sharbat"
        ]
    },
    {
        "user_input": "Black tea, milk, sugar, ice",
        "relevant_recipes": [
            "Chilled Masala Chai",
            "Masala Chai Frappe"
        ]
    },
    {
        "user_input": "Green tea, mint, sugar",
        "relevant_recipes": [
            "Chilled Green Tea"
        ]
    },
    {
        "user_input": "Tea, peach syrup, ice, lemon",
        "relevant_recipes": [
            "Peach Ice Tea"
        ]
    },
    {
        "user_input": "Dried plums (aloo bukhara), sugar, water",
        "relevant_recipes": [
            "Aloo Bukhara Drink"
        ]
    },
    {
        "user_input": "Milk, chocolate syrup, ice",
        "relevant_recipes": [
            "Cold Chocolate Milk"
        ]
    },
    {
        "user_input": "Coconut water, mint, lime juice, sugar",
        "relevant_recipes": [
            "Coconut Mojito"
        ]
    },
    {
        "user_input": "Yogurt, sugar, cardamom, water",
        "relevant_recipes": [
            "Bihari Lassi"
        ]
    },
    {
        "user_input": "Ripe guava, sugar, water",
        "relevant_recipes": [
            "Guava Juice"
        ]
    },
    {
        "user_input": "Tamarind, sugar, soda",
        "relevant_recipes": [
            "Tamarind Fizz"
        ]
    },
    {
        "user_input": "Avocado, milk, honey",
        "relevant_recipes": [
            "Avocado Milkshake"
        ]
    },
    {
        "user_input": "Saffron, milk, sugar",
        "relevant_recipes": [
            "Kesar Doodh"
        ]
    },
    {
        "user_input": "Mint, lime juice, soda",
        "relevant_recipes": [
            "Mint Cooler"
        ]
    },

{
  "user_input": "Coconut, Mint, Yogurt",
  "relevant_recipes": [
    "Coconut Mint Lassi",
    "Berry Coconut Smoothie",
    "Coconut Chia Pudding",
    "Coconut Berry Iced Tea",
    "Almond Coconut Lassi",
    "Coconut Mango Slush",
    "Strawberry Coconut Smoothie",
    "Coconut Prawn Biryani"
  ]
},
{
  "user_input": "Mango, Mint, Yogurt",
  "relevant_recipes": [
    "Mango Mint Ice Tea",
    "Minty Mango Smoothie",
    "Peach Mango Smoothie",
    "Mango Coconut Milkshake",
    "Mango Ginger Mocktail",
    "Sweet Mango Chai",
    "Mango Basil Slush"
  ]
},
{
  "user_input": "Lemon, Ginger, Mint",
  "relevant_recipes": [
    "Spicy Ginger Mint Lemonade",
    "Ginger Lime Cooler",
    "Spicy Ginger Cooler",
    "Spicy Lemonade",
    "Ginger Lemon Iced Tea",
    "Ginger Lychee Cooler",
    "Minty Ginger Lemonade"
  ]
},
{
  "user_input": "Strawberries, Yogurt, Chaat Masala",
  "relevant_recipes": [
    "Strawberry Chaat",
    "Strawberry Chaat Cooler",
    "Blueberry Chaat",
    "Berry Chaat",
    "Green Apple Chaat"
  ]
},
{
  "user_input": "Coconut Milk, Saffron, Sugar",
  "relevant_recipes": [
    "Saffron Coconut Shake",
    "Cardamom Coconut Iced Tea",
    "Coconut Lime Slush",
    "Coconut Pudding"
  ]
},
{
  "user_input": "Mango, Coconut, Lime",
  "relevant_recipes": [
    "Mango Coconut Milkshake",
    "Coconut Mango Slush",
    "Coconut Lime Slush",
    "Mango Ginger Mocktail"
  ]
},
{
  "user_input": "Ginger, Lime, Mint",
  "relevant_recipes": [
    "Spicy Ginger Mint Lemonade",
    "Ginger Lime Cooler",
    "Spicy Ginger Cooler",
    "Minty Ginger Lemonade",
    "Kiwi Lime Sparkler"
  ]
},
{
  "user_input": "Lychee, Coconut Milk, Mint",
  "relevant_recipes": [
    "Lychee Coconut Water",
    "Lychee Mint Iced Tea",
    "Lychee Green Tea",
    "Lychee Smoothie",
    "Lychee Pudding Drink"
  ]
},
{
    "user_input": "Mango, Yogurt, Chaat Masala",
    "relevant_recipes": [
        "Mango Chaat Drink",
        "Sweet Mango Chaat",
        "Mango Mint Slush",
        "Mango Rose Milkshake",
        "Mango Cardamom Lassi",
        "Mango Basil Lemonade"
    ]
},
{
    "user_input": "Coconut, Mint, Yogurt",
    "relevant_recipes": [
        "Coconut Mint Iced Tea",
        "Watermelon Mint Smoothie",
        "Mint Chia Water",
        "Coconut Lychee Smoothie",
        "Tamarind Coconut Slush",
        "Coconut Tamarind Drink"
    ]
},
{
    "user_input": "Lemon, Ginger, Honey",
    "relevant_recipes": [
        "Ginger Honey Lemonade",
        "Ginger Lemon Sparkler",
        "Saffron Ginger Tea",
        "Spicy Ginger Lemonade",
        "Ginger Peach Iced Tea"
    ]
},
{
    "user_input": "Lychee, Yogurt, Rose Syrup",
    "relevant_recipes": [
        "Rose Lychee Lassi",
        "Lychee Mango Sorbet",
        "Lychee Mint Smoothie",
        "Lychee Rose Smoothie",
        "Lychee Basil Lemonade"
    ]
},
{
    "user_input": "Pineapple, Mint, Coconut Milk",
    "relevant_recipes": [
        "Chilled Pineapple Delight",
        "Pineapple Mint Cooler",
        "Coconut Berry Chaat",
        "Coconut Tamarind Drink"
    ]
},
{
    "user_input": "Mango, Coconut, Chaat Masala",
    "relevant_recipes": [
        "Mango Coconut Chaat",
        "Coconut Mango Chaat",
        "Sweet Mango Chaat",
        "Mango Chaat Drink"
    ]
},
{
    "user_input": "Watermelon, Mint, Lime",
    "relevant_recipes": [
        "Watermelon Mint Smoothie",
        "Spiced Watermelon Cooler",
        "Watermelon Rose Slush",
        "Mango Mint Slush"
    ]
},
{
    "user_input": "Papaya, Mint, Yogurt",
    "relevant_recipes": [
        "Papaya Mint Smoothie",
        "Watermelon Mint Smoothie",
        "Lychee Mint Smoothie"
    ]
},
{
    "user_input": "Coconut, Ginger, Milk",
    "relevant_recipes": [
        "Coconut Ginger Shake",
        "Coconut Ginger Limeade",
        "Spiced Coconut Chai",
        "Coconut Tamarind Slush",
        "Coconut Avocado Smoothie",
        "Coconut Lychee Mocktail"
    ]
},
{
    "user_input": "Mint, Lemon Juice, Soda",
    "relevant_recipes": [
        "Mint Lemonade with a Twist",
        "Minty Lychee Cooler",
        "Minty Mango Sparkler",
        "Ginger Mint Cooler",
        "Mint Tamarind Cooler",
        "Minty Pineapple Juice",
        "Minty Watermelon Lemonade"
    ]
},
{
    "user_input": "Tamarind, Sugar, Soda",
    "relevant_recipes": [
        "Pomegranate Ginger Mocktail",
        "Lychee Tamarind Refresher",
        "Spicy Tamarind Limeade",
        "Tamarind Mint Slush",
        "Tamarind Guava Smoothie",
        "Tamarind Lemonade",
        "Tamarind Peach Drink",
        "Tamarind Lychee Cooler"
    ]
},
{
    "user_input": "Strawberries, Yogurt, Chaat Masala",
    "relevant_recipes": [
        "Berry Chaat Mocktail",
        "Watermelon Chaat Delight",
        "Chilled Strawberry Chaat",
        "Sweet Watermelon Chaat"
    ]
},
{
    "user_input": "Lychee, Mint, Soda",
    "relevant_recipes": [
        "Lychee Mint Cooler",
        "Lychee Mint Spritzer",
        "Lychee Mint Delight",
        "Minty Lychee Cooler",
        "Lychee Mint Cooler",
        "Lychee Mint Spritzer"
    ]
},
{
    "user_input": "Coconut, Mixed Berries, Honey",
    "relevant_recipes": [
        "Strawberry Coconut Delight",
        "Coconut Berry Smoothie",
        "Chilled Coconut Berry Smoothie",
        "Coconut Berry Slush",
        "Coconut Berry Fizz"
    ]
},
{
    "user_input": "Mango, Yogurt, Chaat Masala",
    "relevant_recipes": [
        "Chaat Masala Mango Smoothie",
        "Mango Chaat Mocktail",
        "Spicy Mango Lemonade"
    ]
},
{
    "user_input": "Watermelon, Mint, Lime Juice",
    "relevant_recipes": [
        "Minty Watermelon Refresher",
        "Watermelon Mint Cooler",
        "Minty Watermelon Lemonade"
    ]
},
{
    "user_input": "Lychee, Ginger, Soda",
    "relevant_recipes": [
        "Lychee Ginger Fizz",
        "Lychee Ginger Smoothie"
    ]
},
{
    "user_input": "Peach, Yogurt, Chaat Masala",
    "relevant_recipes": [
        "Peach Chaat Delight",
        "Ginger Peach Lemonade"
    ]
},
{
    "user_input": "Coconut, Cardamom, Honey",
    "relevant_recipes": [
        "Spiced Coconut Smoothie",
        "Spiced Coconut Limeade"
    ]
},
{
    "user_input": "Tamarind, Lychee, Sugar",
    "relevant_recipes": [
        "Tamarind Lychee Cooler",
        "Tamarind Lychee Slush"
    ]
}

]









In [73]:
#testing some other metrics:
def evaluate_semantic(test_queries, model, recipe_df, recipe_embeddings, top_n=5):
    metrics = {
        "Semantic Precision": [],
        "Semantic Recall": [],
        "Semantic F1": [],
        "Coverage": [],
        "Jaccard Similarity": []
    }
    
    for query in test_queries:
        user_input = query["user_input"]
        relevant_recipes = query["relevant_recipes"]

        # Encode user input
        user_embedding = model.encode([user_input])
        
        # Compute cosine similarity with all recipes
        similarities = cosine_similarity(user_embedding, recipe_embeddings)[0]
        
        # Get top-N recommended recipes
        top_indices = np.argsort(similarities)[-top_n:][::-1]
        recommended_recipes = recipe_df.iloc[top_indices]
        recommended_embeddings = recipe_embeddings[top_indices]
        
        # Encode relevant recipes
        relevant_embeddings = model.encode(
            recipe_df[recipe_df["Recipe Name"].isin(relevant_recipes)]["combined_text"].tolist()
        )
        
        # Semantic Precision
        semantic_precision = 0
        for rec_emb in recommended_embeddings:
            semantic_precision += max(cosine_similarity([rec_emb], relevant_embeddings)[0])
        semantic_precision /= len(recommended_recipes)
        
        # Semantic Recall
        semantic_recall = 0
        for rel_emb in relevant_embeddings:
            semantic_recall += max(cosine_similarity([rel_emb], recommended_embeddings)[0])
        semantic_recall /= len(relevant_recipes)
        
        # Semantic F1
        if semantic_precision + semantic_recall > 0:
            semantic_f1 = 2 * (semantic_precision * semantic_recall) / (semantic_precision + semantic_recall)
        else:
            semantic_f1 = 0
        
        # Coverage
        user_ingredients = set([i.strip() for i in user_input.split(",")])
        coverage = 0
        for _, recipe in recommended_recipes.iterrows():
            recipe_ingredients = set([i.strip() for i in recipe["Ingredients"].split(",")])
            coverage += len(user_ingredients & recipe_ingredients) / len(user_ingredients)
        coverage /= len(recommended_recipes)
        
        # Jaccard Similarity
        jaccard_similarity = 0
        for _, recipe in recommended_recipes.iterrows():
            recipe_ingredients = set([i.strip() for i in recipe["Ingredients"].split(",")])
            jaccard_similarity += len(user_ingredients & recipe_ingredients) / len(user_ingredients | recipe_ingredients)
        jaccard_similarity /= len(recommended_recipes)
        
        # Append metrics
        metrics["Semantic Precision"].append(semantic_precision)
        metrics["Semantic Recall"].append(semantic_recall)
        metrics["Semantic F1"].append(semantic_f1)
        metrics["Coverage"].append(coverage)
        metrics["Jaccard Similarity"].append(jaccard_similarity)
    
    # Average metrics
    final_metrics = {k: np.mean(v) for k, v in metrics.items()}
    return final_metrics

# Example usage
results = evaluate_semantic(test_queries, model, recipe_df, recipe_embeddings, top_n=5)
print("Evaluation Results:", results)


Evaluation Results: {'Semantic Precision': 0.9030090183019637, 'Semantic Recall': 0.6555806547403336, 'Semantic F1': 0.7472149514445621, 'Coverage': 0.21666666666666667, 'Jaccard Similarity': 0.12333333333333334}


In [54]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Function to calculate Precision, Recall, F1-Score based on similarity threshold
def calculate_metrics_with_similarity(recommended, relevant, recipe_embeddings, threshold=0.7, top_n=3):
    precision_list = []
    recall_list = []
    f1_list = []
    
    # Compare each recommended recipe's embedding to the relevant recipes
    for recommended_recipe in recommended[:top_n]:
        recommended_recipe_embedding = recipe_embeddings[recommended_recipe]
        
        # Calculate similarity with each relevant recipe
        similarities = []
        for relevant_recipe in relevant:
            relevant_recipe_embedding = recipe_embeddings[relevant_recipe]
            sim = cosine_similarity([recommended_recipe_embedding], [relevant_recipe_embedding])[0][0]
            similarities.append(sim)
        
        # Check if similarity is above the threshold
        relevant_matches = sum([1 for sim in similarities if sim >= threshold])
        
        # Calculate Precision and Recall based on the threshold
        precision = relevant_matches / top_n
        recall = relevant_matches / len(relevant)
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)
    
    # Return average precision, recall, and F1
    avg_precision = np.mean(precision_list)
    avg_recall = np.mean(recall_list)
    avg_f1 = np.mean(f1_list)
    
    return avg_precision, avg_recall, avg_f1

# Example test queries with user input and relevant recipes
test_queries = [
    {
        "user_input": "Lychee, Ginger, Soda",
        "relevant_recipes": [
            "Lychee Ginger Fizz",
            "Lychee Ginger Smoothie"
        ]
    },
    {
        "user_input": "Peach, Yogurt, Chaat Masala",
        "relevant_recipes": [
            "Peach Chaat Delight",
            "Ginger Peach Lemonade"
        ]
    },
    {
        "user_input": "Coconut, Cardamom, Honey",
        "relevant_recipes": [
            "Spiced Coconut Smoothie",
            "Spiced Coconut Limeade"
        ]
    },
    {
        "user_input": "Tamarind, Lychee, Sugar",
        "relevant_recipes": [
            "Tamarind Lychee Cooler",
            "Tamarind Lychee Slush"
        ]
    }
]

# Example of generating recommendations from your model
top_n = 3  # Top 3 recommendations

# Initialize results storage
all_precision = []
all_recall = []
all_f1 = []

# Loop through each test query
for test_query in test_queries:
    user_input = test_query["user_input"]
    relevant_recipes = test_query["relevant_recipes"]
    
    # Generate the embedding for user input (assume your model is preloaded)
    user_embedding = model.encode([user_input])

    # Compute similarity scores (assuming you already have recipe embeddings for each recipe)
    similarities = cosine_similarity(user_embedding, recipe_embeddings)

    # Get top-N recommended recipes based on cosine similarity
    top_indices = similarities[0].argsort()[-top_n:][::-1]  # Sort to get top N indices in descending order
    recommended_recipes = recipe_df.iloc[top_indices]['Recipe Name'].tolist()
    
    # Evaluate using the adjusted similarity-based approach
    avg_precision, avg_recall, avg_f1 = calculate_metrics_with_similarity(recommended_recipes, relevant_recipes, recipe_embeddings, top_n=top_n)
    
    # Store the average metrics for each query
    all_precision.append(avg_precision)
    all_recall.append(avg_recall)
    all_f1.append(avg_f1)

# Calculate and print the overall average precision, recall, and F1
print(f"\nAverage Precision@{top_n}: {np.mean(all_precision):.2f}")
print(f"Average Recall@{top_n}: {np.mean(all_recall):.2f}")
print(f"Average F1-Score@{top_n}: {np.mean(all_f1):.2f}")


IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

In [45]:
from sklearn.metrics.pairwise import cosine_similarity

# Test queries based on your data


# Generate recommendations
def generate_recommendations(user_input, recipe_embeddings, recipe_df, top_n):
    # Encode user input
    user_embedding = model.encode([user_input])

    # Compute similarities
    similarities = cosine_similarity(user_embedding, recipe_embeddings)

    # Get top-N indices
    top_indices = similarities[0].argsort()[-top_n:][::-1]

    # Retrieve top-N recipes
    recommendations = recipe_df.iloc[top_indices]['Recipe Name'].tolist()
    return recommendations, similarities[0][top_indices]

# Metrics
def precision_at_n(recommendations, relevant_recipes, n):
    relevant_in_top_n = [rec for rec in recommendations[:n] if rec in relevant_recipes]
    return len(relevant_in_top_n) / n

def recall_at_n(recommendations, relevant_recipes, n):
    relevant_in_top_n = [rec for rec in recommendations[:n] if rec in relevant_recipes]
    return len(relevant_in_top_n) / len(relevant_recipes)

def mean_average_precision(recommendations, relevant_recipes):
    ap_sum = 0
    relevant_found = 0

    for i, rec in enumerate(recommendations):
        if rec in relevant_recipes:
            relevant_found += 1
            ap_sum += relevant_found / (i + 1)
    return ap_sum / len(relevant_recipes) if relevant_recipes else 0

def mean_reciprocal_rank(recommendations, relevant_recipes):
    for i, rec in enumerate(recommendations):
        if rec in relevant_recipes:
            return 1 / (i + 1)
    return 0

# Evaluation function
def evaluate_with_synthetic_inputs(test_queries, recipe_embeddings, recipe_df, top_n=3):
    precision_scores = []
    recall_scores = []
    map_scores = []
    mrr_scores = []

    for query in test_queries:
        user_input = query['user_input']
        relevant_recipes = query['relevant_recipes']

        # Generate recommendations
        recommendations, similarities = generate_recommendations(user_input, recipe_embeddings, recipe_df, top_n)

        # Calculate metrics
        precision_scores.append(precision_at_n(recommendations, relevant_recipes, top_n))
        recall_scores.append(recall_at_n(recommendations, relevant_recipes, top_n))
        map_scores.append(mean_average_precision(recommendations, relevant_recipes))
        mrr_scores.append(mean_reciprocal_rank(recommendations, relevant_recipes))

    # Print results
    print(f"Precision@{top_n}: {sum(precision_scores) / len(precision_scores):.2f}")
    print(f"Recall@{top_n}: {sum(recall_scores) / len(recall_scores):.2f}")
    print(f"Mean Average Precision (MAP): {sum(map_scores) / len(map_scores):.2f}")
    print(f"Mean Reciprocal Rank (MRR): {sum(mrr_scores) / len(mrr_scores):.2f}")

# Evaluate
top_n = 5
evaluate_with_synthetic_inputs(test_queries, recipe_embeddings, recipe_df, top_n)


Precision@5: 0.19
Recall@5: 0.38
Mean Average Precision (MAP): 0.33
Mean Reciprocal Rank (MRR): 0.52


In [42]:
from sklearn.metrics.pairwise import cosine_similarity

# Generate recommendations
def generate_recommendations(user_input, recipe_embeddings, recipe_df, top_n):
    # Encode user input
    user_embedding = model.encode([user_input])

    # Compute similarities
    similarities = cosine_similarity(user_embedding, recipe_embeddings)

    # Get top-N indices
    top_indices = similarities[0].argsort()[-top_n:][::-1]

    # Retrieve top-N recipes
    recommendations = recipe_df.iloc[top_indices]['Recipe Name'].tolist()
    return recommendations, similarities[0][top_indices]

# Metrics
def precision_at_n(recommendations, relevant_recipes, n):
    relevant_in_top_n = [rec for rec in recommendations[:n] if rec in relevant_recipes]
    return len(relevant_in_top_n) / n if n > 0 else 0

def recall_at_n(recommendations, relevant_recipes, n):
    relevant_in_top_n = [rec for rec in recommendations[:n] if rec in relevant_recipes]
    return len(relevant_in_top_n) / len(relevant_recipes) if len(relevant_recipes) > 0 else 0

def mean_average_precision(recommendations, relevant_recipes):
    ap_sum = 0
    relevant_found = 0
    if not relevant_recipes:
        return 0

    for i, rec in enumerate(recommendations):
        if rec in relevant_recipes:
            relevant_found += 1
            ap_sum += relevant_found / (i + 1)
    return ap_sum / len(relevant_recipes) if relevant_recipes else 0

def mean_reciprocal_rank(recommendations, relevant_recipes):
    for i, rec in enumerate(recommendations):
        if rec in relevant_recipes:
            return 1 / (i + 1)
    return 0

# Evaluation function
def evaluate_with_synthetic_inputs(test_queries, recipe_embeddings, recipe_df, top_n=3):
    precision_scores = []
    recall_scores = []
    map_scores = []
    mrr_scores = []

    for query in test_queries:
        user_input = query['user_input']
        relevant_recipes = query['relevant_recipes']

        # Generate recommendations
        recommendations, similarities = generate_recommendations(user_input, recipe_embeddings, recipe_df, top_n)

        # Calculate metrics
        precision_scores.append(precision_at_n(recommendations, relevant_recipes, top_n))
        recall_scores.append(recall_at_n(recommendations, relevant_recipes, top_n))
        map_scores.append(mean_average_precision(recommendations, relevant_recipes))
        mrr_scores.append(mean_reciprocal_rank(recommendations, relevant_recipes))

    # Print results
    print(f"Precision@{top_n}: {sum(precision_scores) / len(precision_scores):.2f}")
    print(f"Recall@{top_n}: {sum(recall_scores) / len(recall_scores):.2f}")
    print(f"Mean Average Precision (MAP): {sum(map_scores) / len(map_scores):.2f}")
    print(f"Mean Reciprocal Rank (MRR): {sum(mrr_scores) / len(mrr_scores):.2f}")

# Evaluate
top_n = 5  # or change this value based on your evaluation needs
evaluate_with_synthetic_inputs(test_queries, recipe_embeddings, recipe_df, top_n)


Precision@5: 0.20
Recall@5: 0.38
Mean Average Precision (MAP): 0.33
Mean Reciprocal Rank (MRR): 0.53


ALL SCORES F1 

In [18]:
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import cosine_similarity

# Test queries based on your data
# Test queries with more than one relevant recipe
test_queries = [
    {
        "user_input": "Beef, Black Pepper, Garlic, Ghee",
        "relevant_recipes": [
            "Beef Namkeen Gosht",
            "Dum Pukht",
            "Beef Pasanda Biryani",
            "Black Pepper Pulao",
            "Spicy Beef Biryani",
            "Beef Haleem Rice",
            "Beef and Lentil Rice",
            "Beef and Potato Pulao"
        ]
    },
    {"user_input": "Flour, Sugar, Butter, Eggs", "relevant_recipes": ["Cake", "Cupcakes", "Butter Cookies"]},
    {"user_input": "Yogurt, Chicken, Spices", "relevant_recipes": ["Chicken Yogurt Curry", "Yogurt Chicken Gravy"]},
    {"user_input": "Lentils, Tomatoes, Spices", "relevant_recipes": ["Lentil Soup", "Dal"]},
    {"user_input": "Potatoes, Chicken, Onions", "relevant_recipes": ["Chicken Masala", "Chicken Potato Curry"]},
    {"user_input": "Milk, Sugar, Rice", "relevant_recipes": ["Kheer", "Rice Pudding"]},
    {"user_input": "Tomatoes, Chicken, Garlic", "relevant_recipes": ["Chicken Curry", "Tomato Chicken"]},
    {"user_input": "Spinach, Paneer, Spices", "relevant_recipes": ["Palak Paneer", "Spinach Curry"]},
    {"user_input": "Flour, Water, Salt, Oil", "relevant_recipes": ["Chapati", "Paratha"]},
    {"user_input": "Bananas, Sugar, Flour", "relevant_recipes": ["Banana Cake", "Banana Bread"]}
]

# Generate recommendations
def generate_recommendations(user_input, recipe_embeddings, recipe_df, top_n=3):
    # Encode user input
    user_embedding = model.encode([user_input])

    # Compute similarities
    similarities = cosine_similarity(user_embedding, recipe_embeddings)

    # Get top-N indices
    top_indices = similarities[0].argsort()[-top_n:][::-1]

    # Retrieve top-N recipes
    recommendations = recipe_df.iloc[top_indices]['Recipe Name'].tolist()
    return recommendations, similarities[0][top_indices]

# Metrics
def precision_at_n(recommendations, relevant_recipes, n):
    relevant_in_top_n = [rec for rec in recommendations[:n] if rec in relevant_recipes]
    return len(relevant_in_top_n) / n

def recall_at_n(recommendations, relevant_recipes, n):
    relevant_in_top_n = [rec for rec in recommendations[:n] if rec in relevant_recipes]
    return len(relevant_in_top_n) / len(relevant_recipes)

def mean_average_precision(recommendations, relevant_recipes):
    ap_sum = 0
    relevant_found = 0

    for i, rec in enumerate(recommendations):
        if rec in relevant_recipes:
            relevant_found += 1
            ap_sum += relevant_found / (i + 1)
    return ap_sum / len(relevant_recipes) if relevant_recipes else 0

def mean_reciprocal_rank(recommendations, relevant_recipes):
    for i, rec in enumerate(recommendations):
        if rec in relevant_recipes:
            return 1 / (i + 1)
    return 0

def f1_at_n(recommendations, relevant_recipes, n):
    # Binary relevance vector for top-N recommendations
    y_true = [1 if rec in relevant_recipes else 0 for rec in recommendations[:n]]
    y_pred = [1] * len(y_true)  # As we're always predicting the top-N recommendations
    
    # Compute F1 score
    return f1_score(y_true, y_pred)

# Evaluation function
def evaluate_with_synthetic_inputs(test_queries, recipe_embeddings, recipe_df, top_n=3):
    precision_scores = []
    recall_scores = []
    map_scores = []
    mrr_scores = []
    f1_scores = []

    for query in test_queries:
        user_input = query['user_input']
        relevant_recipes = query['relevant_recipes']

        # Generate recommendations
        recommendations, similarities = generate_recommendations(user_input, recipe_embeddings, recipe_df, top_n)

        # Calculate metrics
        precision_scores.append(precision_at_n(recommendations, relevant_recipes, top_n))
        recall_scores.append(recall_at_n(recommendations, relevant_recipes, top_n))
        map_scores.append(mean_average_precision(recommendations, relevant_recipes))
        mrr_scores.append(mean_reciprocal_rank(recommendations, relevant_recipes))
        f1_scores.append(f1_at_n(recommendations, relevant_recipes, top_n))

    # Print results
    print(f"Precision@{top_n}: {sum(precision_scores) / len(precision_scores):.2f}")
    print(f"Recall@{top_n}: {sum(recall_scores) / len(recall_scores):.2f}")
    print(f"Mean Average Precision (MAP): {sum(map_scores) / len(map_scores):.2f}")
    print(f"Mean Reciprocal Rank (MRR): {sum(mrr_scores) / len(mrr_scores):.2f}")
    print(f"F1 Score@{top_n}: {sum(f1_scores) / len(f1_scores):.2f}")

# Evaluate
evaluate_with_synthetic_inputs(test_queries, recipe_embeddings, recipe_df, top_n=3)


Precision@3: 0.07
Recall@3: 0.03
Mean Average Precision (MAP): 0.03
Mean Reciprocal Rank (MRR): 0.10
F1 Score@3: 0.08


In [71]:
from sklearn.metrics.pairwise import cosine_similarity

# Example function to handle multi-label matching in categories
def matches_category(selected_category, category_field):
    # Check if the selected category is part of the category field (multi-label)
    return any(selected_category in cat for cat in category_field.split('/'))

# Get ingredients from the user
user_input = input("Enter the ingredients you have (comma-separated): ").strip()

# Generate embedding for user input (using your model, assumed to be pre-loaded)
user_embedding = model.encode([user_input])

# Compute similarity scores (this assumes you have pre-computed recipe embeddings)
similarities = cosine_similarity(user_embedding, recipe_embeddings)

# Get available categories and cuisine regions
available_categories = recipe_df['Category'].unique()
available_cuisines = recipe_df['Cuisine Region'].unique()

# Display available categories and cuisines
print("\nAvailable Categories:")
for category in available_categories:
    print(f"- {category}")

print("\nAvailable Cuisine Regions:")
for cuisine in available_cuisines:
    print(f"- {cuisine}")

# Ask user if they want to narrow down the search by category or cuisine region
category_choice = input("\nWould you like to narrow the search by category? (yes/no): ").strip().lower()
cuisine_choice = input("Would you like to narrow the search by cuisine region? (yes/no): ").strip().lower()

# Filter based on category if user chooses to
if category_choice == 'yes':
    selected_category = input(f"Please choose a category from the above options: ").strip()
    recipe_df = recipe_df[recipe_df['Category'].apply(lambda x: matches_category(selected_category, x))]

# Filter based on cuisine region if user chooses to
if cuisine_choice == 'yes':
    selected_cuisine = input(f"Please choose a cuisine region from the above options: ").strip()
    recipe_df = recipe_df[recipe_df['Cuisine Region'] == selected_cuisine]

# Combine columns again for the filtered recipes
filtered_recipes = recipe_df['Ingredients'] + " | " + recipe_df['Cuisine Region'] + " | " + recipe_df['Cooking Method'] + " | " + recipe_df['Recipe']
filtered_recipe_embeddings = model.encode(filtered_recipes.tolist())

# Compute similarity scores with the filtered recipes
similarities = cosine_similarity(user_embedding, filtered_recipe_embeddings)

# Find the most similar recipe
most_similar_index = similarities[0].argmax()
print("\nBest Match Recipe Name:", recipe_df.iloc[most_similar_index]['Recipe Name'])

# Display top-N matches
top_n = 3
top_indices = similarities[0].argsort()[-top_n:][::-1]  # Sort to get top N indices in descending order

print("\nTop Matches:")
for idx in top_indices:
    recipe_info = recipe_df.iloc[idx]
    print(f"Recipe Name: {recipe_info['Recipe Name']}")
    print(f"  Ingredients: {recipe_info['Ingredients']}")
    print(f"  Cuisine Region: {recipe_info['Cuisine Region']}")
    print(f"  Cooking Method: {recipe_info['Cooking Method']}")
    print(f"  Recipe: {recipe_info['Recipe']}")
    print(f"  Category: {recipe_info['Category']}\n")


Enter the ingredients you have (comma-separated):  chicken,spices,garlic,onions,rice



Available Categories:
- Rice
- Meat
- Vegetarian
- Street Food
- Dessert
- BBQ-GRILL
- Seafood
- Breakfast
- Drinks
- Rice/Vegetarian
- Breakfast/Street Food
- Meat/Rice
- Meat/Street Food
- Breakfast/Rice
- Rice/Street Food
- Breakfast/Meat/Rice
- Breakfast/Drinks
- Rice/Street Food/Vegetarian
- BBQ-GRILL/Street Food
- Dessert/Drinks
- Breakfast/Rice/Vegetarian
- Dessert/Drinks/Street Food
- Seafood/Street Food
- Rice/Seafood/Street Food
- BBQ-GRILL/Seafood
- Dessert/Street Food
- Breakfast/Dessert/Street Food
- Breakfast/Dessert
- Breakfast/Drinks/Street Food
- Breakfast/Vegetarian
- Rice/Seafood
- Breakfast/Seafood
- Dessert/Rice

Available Cuisine Regions:
- Punjab
- Sindh
- Kashmir
- Khyber Pakhtunkhwa
- Coastal Region
- Hyderabad
- Fusion
- Balochistan



Would you like to narrow the search by category? (yes/no):  no
Would you like to narrow the search by cuisine region? (yes/no):  no



Best Match Recipe Name: Vegetable Pulao

Top Matches:
Recipe Name: Vegetable Pulao
  Ingredients: Rice, Mixed Vegetables (Carrots, Peas, Beans), Whole Spices, Onions, Ginger, Garlic, Ghee
  Cuisine Region: Fusion
  Cooking Method: Boiling, Simmering
  Recipe: Cook mixed vegetables with spices. Boil rice separately and mix. Serve with salad and raita.
  Category: Rice/Vegetarian

Recipe Name: Spicy Chicken and Rice Bowl
  Ingredients: Chicken, rice, spices, vegetables
  Cuisine Region: Fusion
  Cooking Method: Boiled
  Recipe: Cook spiced chicken and serve over rice with sautéed vegetables.
  Category: Breakfast

Recipe Name: Grilled Chicken & Cauliflower Rice
  Ingredients: Grilled chicken, cauliflower, spices, herbs
  Cuisine Region: Fusion
  Cooking Method: BBQ
  Recipe: Grill chicken, serve over cauliflower rice with spices.
  Category: BBQ-GRILL



In [29]:
from sklearn.metrics.pairwise import cosine_similarity

# Test queries based on your data


# Generate recommendations
def generate_recommendations(user_input, recipe_embeddings, recipe_df, top_n=3):
    # Encode user input
    user_embedding = model.encode([user_input])

    # Compute similarities
    similarities = cosine_similarity(user_embedding, recipe_embeddings)

    # Get top-N indices
    top_indices = similarities[0].argsort()[-top_n:][::-1]

    # Retrieve top-N recipes
    recommendations = recipe_df.iloc[top_indices]['Recipe Name'].tolist()
    return recommendations, similarities[0][top_indices]

# Metrics
def precision_at_n(recommendations, relevant_recipes, n):
    relevant_in_top_n = [rec for rec in recommendations[:n] if rec in relevant_recipes]
    return len(relevant_in_top_n) / n

def recall_at_n(recommendations, relevant_recipes, n):
    relevant_in_top_n = [rec for rec in recommendations[:n] if rec in relevant_recipes]
    return len(relevant_in_top_n) / len(relevant_recipes)

def mean_average_precision(recommendations, relevant_recipes):
    ap_sum = 0
    relevant_found = 0

    for i, rec in enumerate(recommendations):
        if rec in relevant_recipes:
            relevant_found += 1
            ap_sum += relevant_found / (i + 1)
    return ap_sum / len(relevant_recipes) if relevant_recipes else 0

def mean_reciprocal_rank(recommendations, relevant_recipes):
    for i, rec in enumerate(recommendations):
        if rec in relevant_recipes:
            return 1 / (i + 1)
    return 0

# Evaluation function
def evaluate_with_synthetic_inputs(test_queries, recipe_embeddings, recipe_df, top_n=3):
    precision_scores = []
    recall_scores = []
    map_scores = []
    mrr_scores = []

    for query in test_queries:
        user_input = query['user_input']
        relevant_recipes = query['relevant_recipes']

        # Generate recommendations
        recommendations, similarities = generate_recommendations(user_input, recipe_embeddings, recipe_df, top_n)

        # Calculate metrics
        precision_scores.append(precision_at_n(recommendations, relevant_recipes, top_n))
        recall_scores.append(recall_at_n(recommendations, relevant_recipes, top_n))
        map_scores.append(mean_average_precision(recommendations, relevant_recipes))
        mrr_scores.append(mean_reciprocal_rank(recommendations, relevant_recipes))

    # Print results
    print(f"Precision@{top_n}: {sum(precision_scores) / len(precision_scores):.2f}")
    print(f"Recall@{top_n}: {sum(recall_scores) / len(recall_scores):.2f}")
    print(f"Mean Average Precision (MAP): {sum(map_scores) / len(map_scores):.2f}")
    print(f"Mean Reciprocal Rank (MRR): {sum(mrr_scores) / len(mrr_scores):.2f}")

# Evaluate
evaluate_with_synthetic_inputs(test_queries, recipe_embeddings, recipe_df, top_n=3)


Precision@3: 0.27
Recall@3: 0.34
Mean Average Precision (MAP): 0.31
Mean Reciprocal Rank (MRR): 0.52


Different Types of Evaluation

In [70]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Precision@N based on semantic similarity
def precision_at_n_semantic(recommendations, relevant_recipes, recipe_embeddings, recipe_df, n):
    # Get indices of recommended recipes
    recommended_indices = [
        recipe_df.index[recipe_df['Recipe Name'] == rec].tolist()[0]
        for rec in recommendations[:n]
        if not recipe_df.index[recipe_df['Recipe Name'] == rec].empty
    ]
    recommended_embeddings = recipe_embeddings[recommended_indices]
    
    # Get indices of relevant recipes
    relevant_indices = [
        recipe_df.index[recipe_df['Recipe Name'] == rel].tolist()[0]
        for rel in relevant_recipes
        if not recipe_df.index[recipe_df['Recipe Name'] == rel].empty
    ]
    relevant_embeddings = recipe_embeddings[relevant_indices]

    # Handle edge cases
    if not len(recommended_embeddings) or not len(relevant_embeddings):
        return 0

    # Compute cosine similarity between recommended and relevant embeddings
    similarity_matrix = cosine_similarity(recommended_embeddings, relevant_embeddings)

    # Precision is the average similarity score
    return similarity_matrix.max(axis=1).mean()


# Recall@N based on semantic similarity
# Recall@N based on semantic similarity
def recall_at_n_semantic(recommendations, relevant_recipes, recipe_embeddings, recipe_df, n):
    # Get indices of recommended recipes
    recommended_indices = [
        recipe_df.index[recipe_df['Recipe Name'] == rec].tolist()[0]
        for rec in recommendations[:n]
        if not recipe_df.index[recipe_df['Recipe Name'] == rec].empty
    ]
    recommended_embeddings = recipe_embeddings[recommended_indices]
    
    # Get indices of relevant recipes
    relevant_indices = [
        recipe_df.index[recipe_df['Recipe Name'] == rel].tolist()[0]
        for rel in relevant_recipes
        if not recipe_df.index[recipe_df['Recipe Name'] == rel].empty
    ]
    relevant_embeddings = recipe_embeddings[relevant_indices]

    # Handle edge cases (no recommendations or no relevant recipes)
    if not len(recommended_embeddings) or not len(relevant_embeddings):
        return 0

    # Compute cosine similarity between recommended and relevant embeddings
    similarity_matrix = cosine_similarity(recommended_embeddings, relevant_embeddings)
    
    # Check if recommended recipes are relevant (we use a threshold for relevance)
    threshold = 0.7  # You can adjust this threshold
    matched = (similarity_matrix.max(axis=1) >= threshold).sum()

    # Recall: Relevant items retrieved / Total relevant items
    return matched / min(len(relevant_recipes), n)  # Ensure it's bounded by total relevant recipes or n


# Diversity score based on pairwise dissimilarity
def diversity_score(recommendations, recipe_embeddings, recipe_df):
    recommended_indices = [
        recipe_df.index[recipe_df['Recipe Name'] == rec].tolist()[0]
        for rec in recommendations
        if not recipe_df.index[recipe_df['Recipe Name'] == rec].empty
    ]
    recommended_embeddings = recipe_embeddings[recommended_indices]
    
    # Handle edge cases
    if len(recommended_embeddings) < 2:
        return 0  # No diversity for fewer than 2 recommendations

    # Compute pairwise cosine similarity
    similarity_matrix = cosine_similarity(recommended_embeddings)
    
    # Diversity is 1 - average similarity (excluding diagonal)
    diversity = 1 - np.mean(similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)])
    return diversity


# Evaluation function
def evaluate_with_semantic_metrics(test_queries, recipe_embeddings, recipe_df, top_n=5):
    precision_scores = []
    recall_scores = []
    diversity_scores = []
    all_recommendations = []

    for query in test_queries:
        user_input = query['user_input']
        relevant_recipes = query['relevant_recipes']

        # Generate recommendations
        user_embedding = model.encode([user_input])
        similarities = cosine_similarity(user_embedding, recipe_embeddings)
        top_indices = similarities[0].argsort()[-top_n:][::-1]  # Top-N indices
        recommendations = recipe_df.iloc[top_indices]['Recipe Name'].tolist()
        all_recommendations.extend(recommendations)

        # Calculate semantic precision and recall
        precision_scores.append(precision_at_n_semantic(recommendations, relevant_recipes, recipe_embeddings, recipe_df, top_n))
        recall_scores.append(recall_at_n_semantic(recommendations, relevant_recipes, recipe_embeddings, recipe_df, top_n))
        
        # Calculate diversity
        diversity_scores.append(diversity_score(recommendations, recipe_embeddings, recipe_df))

    # Calculate overall metrics
    semantic_precision = np.mean(precision_scores)
    semantic_recall = np.mean(recall_scores)
    avg_diversity = np.mean(diversity_scores)

    # Print results
    print(f"Semantic Precision@{top_n}: {semantic_precision:.2f}")
    print(f"Semantic Recall@{top_n}: {semantic_recall:.2f}")
    print(f"Diversity: {avg_diversity:.2f}")

    return {
        "Semantic Precision": semantic_precision,
        "Semantic Recall": semantic_recall,
        "Diversity": avg_diversity
    }


# Example usage
# Assume `test_queries`, `recipe_embeddings`, and `recipe_df` are already defined
metrics = evaluate_with_semantic_metrics(test_queries, recipe_embeddings, recipe_df, top_n=5)


Semantic Precision@5: 0.90
Semantic Recall@5: 2.50
Diversity: 0.14


In [67]:
evaluate_with_semantic_metrics(test_queries, recipe_embeddings, recipe_df, top_n=5)


Semantic Precision@5: 0.90
Semantic Recall@5: 2.50
Diversity: 0.14
